<a href="https://colab.research.google.com/github/barr92/public-house-price-data/blob/main/Auction%20Data%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signals BI — Auction Pipeline
**Project:** `signalsbi-new` | **Dataset:** `auctions` (prod) / `auctions_dev` (dev)

## How to use
| Scenario | Settings |
|---|---|
| First ever run | `ENV = 'dev'`, `FIRST_RUN = True` |
| Monthly update | `ENV = 'prod'`, `FIRST_RUN = False` |
| Promote dev → prod | Run Cell 8 with `PROMOTE = True` |

## Tables produced
| Table | Description |
|---|---|
| `ingested_files` | Log of every GCS file processed — prevents re-ingestion |
| `auctions_raw` | Append-only, every row from every scrape, all sources |
| `auctions_status_log` | One row per status change per lot — the changelog |
| `auctions_clean` | One row per lot, latest status, fully normalised |
| `auction_sector_monthly` | Monthly aggregation per postcode sector |
| `auction_sector_summary` | All-time summary per postcode sector |

## 0. Apify webhook trigger (future use)
Commented out — will trigger Apify scraper runs automatically once webhook is configured.

In [ ]:
# ── FUTURE: Apify webhook trigger ──────────────────────────────────────────
# Uncomment when Apify webhook is configured.
# This will trigger scraper runs and wait for completion before ingesting.
#
# import requests
#
# APIFY_TOKEN = 'your_apify_token_here'
# SCRAPERS = {
#     'auction_house': 'your_auction_house_actor_id',
#     'savills':       'your_savills_actor_id',
# }
#
# def trigger_scraper(actor_id):
#     url = f'https://api.apify.com/v2/acts/{actor_id}/runs?token={APIFY_TOKEN}'
#     resp = requests.post(url, json={'memory': 4096})
#     resp.raise_for_status()
#     run_id = resp.json()['data']['id']
#     print(f'Started run {run_id} for {actor_id}')
#     return run_id
#
# for name, actor_id in SCRAPERS.items():
#     trigger_scraper(actor_id)
# ─────────────────────────────────────────────────────────────────────────────

print('Apify trigger skipped (manual mode)')

## 1. Install & import

In [1]:
# !pip install google-cloud-bigquery google-cloud-bigquery-storage google-cloud-storage pandas db-dtypes --quiet

from google.colab import auth
auth.authenticate_user()

import pandas as pd
import re
from datetime import datetime, timezone
from google.cloud import bigquery, storage

print('Imports OK')

Imports OK


## 2. Configuration
**Only cell you need to change between runs.**

In [2]:
# ── CHANGE THESE BETWEEN RUNS ────────────────────────────────────────────────
ENV       = 'dev'   # 'dev' or 'prod'
FIRST_RUN = True    # True = ingest all GCS files | False = only new files
# ─────────────────────────────────────────────────────────────────────────────

PROJECT   = 'signalsbi-new'
DATASETS  = {'prod': 'auctions', 'dev': 'auctions_dev'}
DATASET   = DATASETS[ENV]
D         = f'`{PROJECT}.{DATASET}`'
GCS_BUCKET  = 'signals-new'
GCS_PREFIX  = 'property-auctions/raw/'

# Sources to skip entirely
SKIP_SOURCES = ['sdl', 'SDL']

# Status precedence for lifecycle tracking
# Higher number = more terminal/authoritative status
STATUS_PRECEDENCE = {
    'ACTIVE':     1,
    'UPCOMING':   2,
    'UNKNOWN':    3,
    'POSTPONED':  4,
    'WITHDRAWN':  5,
    'UNSOLD':     6,
    'SOLD_AFTER': 7,
    'SOLD_PRIOR': 8,
    'SOLD':       9,
}

bq     = bigquery.Client(project=PROJECT, location='EU')
gcs    = storage.Client(project=PROJECT)

print(f'Environment : {ENV.upper()}')
print(f'First run   : {FIRST_RUN}')
print(f'Dataset     : {PROJECT}.{DATASET}')

Environment : DEV
First run   : True
Dataset     : signalsbi-new.auctions_dev


## 3. Create dataset and tables
`CREATE TABLE IF NOT EXISTS` — safe to rerun, no-op if already exists.

In [3]:
# Create dataset
ds = bigquery.Dataset(f'{PROJECT}.{DATASET}')
ds.location = 'EU'
bq.create_dataset(ds, exists_ok=True)
print(f'Dataset `{PROJECT}.{DATASET}` ready.')

ddls = [

# 1. ingested_files — tracks which GCS files have been processed
f'''
CREATE TABLE IF NOT EXISTS {D}.ingested_files (
  file_name       STRING NOT NULL,
  gcs_path        STRING,
  source          STRING,
  row_count       INT64,
  ingested_at     TIMESTAMP
)
''',

# 2. auctions_raw — append-only, every row from every scrape
f'''
CREATE TABLE IF NOT EXISTS {D}.auctions_raw (
  -- identity
  signals_bi_id       STRING,
  lot_id_source       STRING,
  lot_id_native       STRING,
  lot_number          STRING,
  source              STRING,
  auction_house       STRING,
  branch              STRING,
  -- location
  address_full        STRING,
  postcode            STRING,
  postcode_sector     STRING,
  postcode_district   STRING,
  town                STRING,
  county              STRING,
  region              STRING,
  country             STRING,
  latitude            FLOAT64,
  longitude           FLOAT64,
  -- property
  title               STRING,
  property_type       STRING,
  property_type_norm  STRING,
  bedrooms            INT64,
  tenure              STRING,
  epc_rating          STRING,
  sq_ft               FLOAT64,
  sq_m                FLOAT64,
  -- auction
  auction_date        DATE,
  auction_ended_at    TIMESTAMP,
  sale_start_date     DATE,
  sale_end_date       DATE,
  auction_id          STRING,
  auction_slug        STRING,
  -- pricing
  guide_price_low     INT64,
  guide_price_high    INT64,
  guide_price_mid     INT64,
  guide_price_text    STRING,
  is_starting_bid     BOOL,
  sold_price          INT64,
  sold_prior          BOOL,
  sold_after          BOOL,
  estimate_perf_pct   FLOAT64,
  -- status
  status              STRING,
  status_precedence   INT64,
  -- meta
  listing_url         STRING,
  page_type           STRING,
  data_version        FLOAT64,
  last_updated_at     TIMESTAMP,
  scraped_at          TIMESTAMP,
  ingested_at         TIMESTAMP,
  file_name           STRING
)
PARTITION BY auction_date
CLUSTER BY postcode_sector, source, status
OPTIONS (require_partition_filter = false)
''',

# 3. auctions_status_log — one row per status change per lot
f'''
CREATE TABLE IF NOT EXISTS {D}.auctions_status_log (
  signals_bi_id       STRING,
  previous_status     STRING,
  new_status          STRING,
  previous_precedence INT64,
  new_precedence      INT64,
  changed_at          TIMESTAMP,
  file_name           STRING,
  source              STRING
)
''',

# 4. auctions_clean — one row per lot, latest status
f'''
CREATE TABLE IF NOT EXISTS {D}.auctions_clean (
  signals_bi_id       STRING,
  source              STRING,
  auction_house       STRING,
  address_full        STRING,
  postcode            STRING,
  postcode_sector     STRING,
  postcode_district   STRING,
  town                STRING,
  county              STRING,
  region              STRING,
  latitude            FLOAT64,
  longitude           FLOAT64,
  property_type       STRING,
  property_type_norm  STRING,
  bedrooms            INT64,
  tenure              STRING,
  auction_date        DATE,
  guide_price_low     INT64,
  guide_price_high    INT64,
  guide_price_mid     INT64,
  sold_price          INT64,
  sold_prior          BOOL,
  sold_after          BOOL,
  estimate_perf_pct   FLOAT64,
  premium_over_guide_pct FLOAT64,
  status              STRING,
  status_precedence   INT64,
  is_sold             BOOL,
  listing_url         STRING,
  first_seen_at       TIMESTAMP,
  last_updated_at     TIMESTAMP,
  refreshed_at        TIMESTAMP
)
CLUSTER BY postcode_sector, status, source
''',

# 5. auction_sector_monthly
f'''
CREATE TABLE IF NOT EXISTS {D}.auction_sector_monthly (
  postcode_sector         STRING,
  postcode_district       STRING,
  year_month              STRING,
  total_lots              INT64,
  total_sold              INT64,
  total_unsold            INT64,
  total_withdrawn         INT64,
  sell_through_rate_pct   FLOAT64,
  avg_guide_price         INT64,
  avg_sold_price          INT64,
  median_sold_price       INT64,
  avg_premium_over_guide  FLOAT64,
  sources                 STRING,
  refreshed_at            TIMESTAMP
)
CLUSTER BY postcode_sector
''',

# 6. auction_sector_summary
f'''
CREATE TABLE IF NOT EXISTS {D}.auction_sector_summary (
  postcode_sector         STRING,
  postcode_district       STRING,
  region                  STRING,
  total_lots              INT64,
  total_sold              INT64,
  total_unsold            INT64,
  total_withdrawn         INT64,
  sell_through_rate_pct   FLOAT64,
  avg_guide_price         INT64,
  avg_sold_price          INT64,
  median_sold_price       INT64,
  min_sold_price          INT64,
  max_sold_price          INT64,
  avg_premium_over_guide  FLOAT64,
  earliest_auction        DATE,
  latest_auction          DATE,
  sources                 STRING,
  refreshed_at            TIMESTAMP
)
CLUSTER BY postcode_sector
'''
]

for ddl in ddls:
    bq.query(ddl).result()

print('All tables ready.')

Dataset `signalsbi-new.auctions_dev` ready.
All tables ready.


## 4. Discover and ingest GCS files
Scans the GCS bucket, skips already-ingested files and SDL sources.

In [4]:
PROP_TYPE_MAP = {
    'flat':            'Flat',
    'apartment':       'Flat',
    'maisonette':      'Flat',
    'studio':          'Flat',
    'house':           'House',
    'detached':        'Detached',
    'semi-detached':   'Semi-Detached',
    'terraced':        'Terraced',
    'end of terrace':  'Terraced',
    'land':            'Land',
    'commercial':      'Commercial',
    'unknown':         None,
}

def normalise_property_type(pt):
    if not pt or str(pt).lower() in ['nan', 'none', '']:
        return None
    return PROP_TYPE_MAP.get(str(pt).lower().strip(), str(pt).strip())

def parse_date(val):
    if not val or str(val).lower() in ['nan', 'none', '']:
        return None
    for fmt in ['%d/%m/%Y', '%Y-%m-%d', '%Y-%m-%dT%H:%M:%S.%fZ', '%Y-%m-%dT%H:%M:%SZ']:
        try:
            from datetime import date
            return datetime.strptime(str(val)[:19], fmt[:len(str(val)[:10])]).date()
        except:
            continue
    return None

def safe_int(val):
    try: return int(float(val)) if val and str(val).lower() not in ['nan','none',''] else None
    except: return None

def safe_float(val):
    try: return float(val) if val and str(val).lower() not in ['nan','none',''] else None
    except: return None

def safe_bool(val):
    if isinstance(val, bool): return val
    if str(val).lower() in ['true','1','yes']: return True
    if str(val).lower() in ['false','0','no']: return False
    return None

def is_sdl(filename, df):
    fname_lower = filename.lower()
    if any(s.lower() in fname_lower for s in ['sdl', 'sdl auctions']):
        return True
    if 'source' in df.columns:
        sources = df['source'].dropna().str.lower().unique()
        if all('sdl' in s for s in sources):
            return True
    return False

def normalise_row(row, file_name, ingested_at):
    r = dict(row)
    g = lambda k: r.get(k)

    guide_low  = safe_int(g('guidePriceLow'))
    guide_high = safe_int(g('guidePriceHigh'))
    guide_mid  = int((guide_low + guide_high) / 2) if guide_low and guide_high else (guide_low or guide_high)

    status = str(g('status') or 'UNKNOWN').upper().strip()
    prec   = STATUS_PRECEDENCE.get(status, 0)

    sold_price = safe_int(g('soldPrice'))

    # auction_date: try auctionDate then saleEndDate
    auction_date = parse_date(g('auctionDate')) or parse_date(g('saleEndDate'))

    return {
        'signals_bi_id':      str(g('signalsBiId') or ''),
        'lot_id_source':      str(g('lotIdSource') or ''),
        'lot_id_native':      str(g('lotId') or ''),
        'lot_number':         str(g('lotNumber') or ''),
        'source':             str(g('source') or ''),
        'auction_house':      str(g('auctionHouse') or ''),
        'branch':             str(g('branch') or ''),
        'address_full':       str(g('addressFull') or ''),
        'postcode':           str(g('postcode') or '').upper().strip(),
        'postcode_sector':    str(g('postcodeSector') or '').upper().strip(),
        'postcode_district':  str(g('postcodeDistrict') or '').upper().strip(),
        'town':               str(g('town') or ''),
        'county':             str(g('county') or ''),
        'region':             str(g('region') or ''),
        'country':            str(g('country') or ''),
        'latitude':           safe_float(g('latitude')),
        'longitude':          safe_float(g('longitude')),
        'title':              str(g('title') or ''),
        'property_type':      str(g('propertyType') or ''),
        'property_type_norm': normalise_property_type(g('propertyType')),
        'bedrooms':           safe_int(g('bedrooms')),
        'tenure':             str(g('tenure') or ''),
        'epc_rating':         str(g('epcRating') or ''),
        'sq_ft':              safe_float(g('sqFtFloorArea')),
        'sq_m':               safe_float(g('sqMFloorArea')),
        'auction_date':       auction_date,
        'auction_ended_at':   pd.to_datetime(g('auctionEndedAt'), errors='coerce', utc=True),
        'sale_start_date':    parse_date(g('saleStartDate')),
        'sale_end_date':      parse_date(g('saleEndDate')),
        'auction_id':         str(g('auctionId') or ''),
        'auction_slug':       str(g('auctionSlug') or ''),
        'guide_price_low':    guide_low,
        'guide_price_high':   guide_high,
        'guide_price_mid':    guide_mid,
        'guide_price_text':   str(g('guidePriceText') or ''),
        'is_starting_bid':    safe_bool(g('isStartingBid')),
        'sold_price':         sold_price,
        'sold_prior':         safe_bool(g('soldPrior')),
        'sold_after':         safe_bool(g('soldAfter')),
        'estimate_perf_pct':  safe_float(g('estimatePerformancePct')),
        'status':             status,
        'status_precedence':  prec,
        'listing_url':        str(g('listingUrl') or ''),
        'page_type':          str(g('pageType') or ''),
        'data_version':       safe_float(g('dataVersion')),
        'last_updated_at':    pd.to_datetime(g('lastUpdatedAt'), errors='coerce', utc=True),
        'scraped_at':         pd.to_datetime(g('scrapedAt'), errors='coerce', utc=True),
        'ingested_at':        ingested_at,
        'file_name':          file_name,
    }

def get_ingested_files():
    try:
        result = bq.query(f'SELECT file_name FROM {D}.ingested_files').to_dataframe()
        return set(result['file_name'].tolist())
    except:
        return set()

def log_ingested_file(file_name, gcs_path, source, row_count, ingested_at):
    df = pd.DataFrame([{
        'file_name': file_name, 'gcs_path': gcs_path,
        'source': source, 'row_count': row_count, 'ingested_at': ingested_at
    }])
    bq.load_table_from_dataframe(df, f'{PROJECT}.{DATASET}.ingested_files',
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')).result()

# ── Main ingest loop ──────────────────────────────────────────────────────────
bucket = gcs.bucket(GCS_BUCKET)
blobs  = list(bucket.list_blobs(prefix=GCS_PREFIX))
already_ingested = set() if FIRST_RUN else get_ingested_files()

print(f'Found {len(blobs)} files in GCS. Already ingested: {len(already_ingested)}')

total_rows = 0
for blob in blobs:
    file_name = blob.name.split('/')[-1]
    if not file_name.endswith('.csv'):
        continue
    if file_name in already_ingested:
        print(f'  SKIP (already ingested): {file_name}')
        continue

    # Download
    print(f'  Reading: {file_name} ...', end=' ', flush=True)
    content = blob.download_as_bytes()
    df_raw = pd.read_csv(__import__('io').BytesIO(content), dtype=str, low_memory=False)
    print(f'{len(df_raw):,} rows')

    # Skip SDL
    if is_sdl(file_name, df_raw):
        print(f'  SKIP (SDL source): {file_name}')
        continue

    # Normalise
    ingested_at = datetime.now(timezone.utc)
    rows = [normalise_row(row, file_name, ingested_at) for _, row in df_raw.iterrows()]
    df_norm = pd.DataFrame(rows)

    # Remove rows with no signals_bi_id
    df_norm = df_norm[df_norm['signals_bi_id'].str.len() > 0]

    # Append to auctions_raw
    source_name = df_norm['source'].iloc[0] if len(df_norm) else 'unknown'
    bq.load_table_from_dataframe(
        df_norm, f'{PROJECT}.{DATASET}.auctions_raw',
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
    ).result()

    # Log file as ingested
    log_ingested_file(file_name, blob.name, source_name, len(df_norm), ingested_at)
    total_rows += len(df_norm)
    print(f'  Ingested {len(df_norm):,} rows from {file_name}')

print(f'\nTotal rows added to auctions_raw: {total_rows:,}')

Found 9 files in GCS. Already ingested: 0
  Reading: SDL Auctions 2026-03-01.csv ... 537 rows
  SKIP (SDL source): SDL Auctions 2026-03-01.csv
  Reading: Savills 2026-03-01.csv ... 1,229 rows
  Ingested 1,229 rows from Savills 2026-03-01.csv
  Reading: auction house 2026-02-27.csv ... 14,787 rows
  Ingested 14,787 rows from auction house 2026-02-27.csv
  Reading: auctions scrape ah 2026-03-16.csv ... 15,000 rows
  Ingested 15,000 rows from auctions scrape ah 2026-03-16.csv
  Reading: dataset_auction-house-uk---web-scraper_2026-05-30_20-44-51-771 (1).csv ... 16,027 rows
  Ingested 16,027 rows from dataset_auction-house-uk---web-scraper_2026-05-30_20-44-51-771 (1).csv
  Reading: dataset_savills-property-auctions-scraper_2026-05-30_23-46-28-551.csv ... 661 rows
  Ingested 661 rows from dataset_savills-property-auctions-scraper_2026-05-30_23-46-28-551.csv
  Reading: savills auction scrape  2026-03-16.csv ... 1,276 rows
  Ingested 1,276 rows from savills auction scrape  2026-03-16.csv
  Rea

## 5. Build status changelog
Detects status changes per lot and logs them to `auctions_status_log`.

In [5]:
status_log_sql = f'''
INSERT INTO {D}.auctions_status_log
  (signals_bi_id, previous_status, new_status, previous_precedence,
   new_precedence, changed_at, file_name, source)

WITH ordered AS (
  SELECT
    signals_bi_id, status, status_precedence, ingested_at, file_name, source,
    LAG(status) OVER (PARTITION BY signals_bi_id ORDER BY ingested_at)          AS prev_status,
    LAG(status_precedence) OVER (PARTITION BY signals_bi_id ORDER BY ingested_at) AS prev_precedence
  FROM {D}.auctions_raw
  WHERE signals_bi_id IS NOT NULL AND signals_bi_id != ''
),
changes AS (
  SELECT *
  FROM ordered
  WHERE prev_status IS NOT NULL
    AND status != prev_status
),
already_logged AS (
  SELECT signals_bi_id, new_status, changed_at
  FROM {D}.auctions_status_log
)
SELECT
  c.signals_bi_id,
  c.prev_status        AS previous_status,
  c.status             AS new_status,
  c.prev_precedence    AS previous_precedence,
  c.status_precedence  AS new_precedence,
  c.ingested_at        AS changed_at,
  c.file_name,
  c.source
FROM changes c
LEFT JOIN already_logged al
  ON c.signals_bi_id = al.signals_bi_id
  AND c.status = al.new_status
  AND c.ingested_at = al.changed_at
WHERE al.signals_bi_id IS NULL
'''

print('Building status changelog...')
bq.query(status_log_sql).result()
n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.auctions_status_log').to_dataframe().iloc[0]['n']
print(f'auctions_status_log: {n:,} change events total')

Building status changelog...
auctions_status_log: 634 change events total


## 6. Build `auctions_clean`
One row per lot using highest-precedence status. Calculates premium over guide.

In [7]:
clean_sql = f'''
CREATE OR REPLACE TABLE {D}.auctions_clean AS

WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY signals_bi_id
      ORDER BY status_precedence DESC, ingested_at DESC
    ) AS rn
  FROM {D}.auctions_raw
  WHERE signals_bi_id IS NOT NULL AND signals_bi_id != ''
),
latest AS (
  SELECT * FROM ranked WHERE rn = 1
),
first_seen AS (
  SELECT signals_bi_id, MIN(ingested_at) AS first_seen_at
  FROM {D}.auctions_raw
  GROUP BY 1
)
SELECT
  l.signals_bi_id,
  l.source,
  l.auction_house,
  l.address_full,
  l.postcode,
  l.postcode_sector,
  l.postcode_district,
  l.town,
  l.county,
  l.region,
  l.latitude,
  l.longitude,
  l.property_type,
  l.property_type_norm,
  l.bedrooms,
  l.tenure,
  l.auction_date,
  l.guide_price_low,
  l.guide_price_high,
  l.guide_price_mid,
  l.sold_price,
  l.sold_prior,
  l.sold_after,
  l.estimate_perf_pct,
  ROUND(SAFE_DIVIDE(l.sold_price - l.guide_price_mid, l.guide_price_mid) * 100, 1)
    AS premium_over_guide_pct,
  l.status,
  l.status_precedence,
  l.status IN ('SOLD', 'SOLD_PRIOR', 'SOLD_AFTER') AS is_sold,
  l.listing_url,
  f.first_seen_at,
  l.last_updated_at,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM latest l
JOIN first_seen f USING (signals_bi_id)
'''

print('Building auctions_clean...')
bq.query(clean_sql).result()
n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.auctions_clean').to_dataframe().iloc[0]['n']
print(f'auctions_clean: {n:,} unique lots')

Building auctions_clean...
auctions_clean: 21,312 unique lots


## 7. Build aggregation tables

In [10]:
def run_query(name, sql):
    print(f'  Refreshing {name}...', end=' ', flush=True)
    bq.query(sql).result()
    print('done')

run_query('auction_sector_monthly', f'''
CREATE OR REPLACE TABLE {D}.auction_sector_monthly AS
SELECT
  postcode_sector,
  postcode_district,
  FORMAT_DATE('%Y-%m', auction_date)          AS year_month,
  COUNT(*)                                    AS total_lots,
  COUNTIF(is_sold)                            AS total_sold,
  COUNTIF(status = 'UNSOLD')                  AS total_unsold,
  COUNTIF(status = 'WITHDRAWN')               AS total_withdrawn,
  ROUND(SAFE_DIVIDE(COUNTIF(is_sold), COUNT(*)) * 100, 1) AS sell_through_rate_pct,
  CAST(ROUND(AVG(guide_price_mid), 0) AS INT64)  AS avg_guide_price,
  CAST(ROUND(AVG(IF(is_sold, sold_price, NULL)), 0) AS INT64) AS avg_sold_price,
  CAST(APPROX_QUANTILES(IF(is_sold, sold_price, NULL), 100)[OFFSET(50)] AS INT64) AS median_sold_price,
  ROUND(AVG(IF(is_sold, premium_over_guide_pct, NULL)), 1) AS avg_premium_over_guide,
  STRING_AGG(DISTINCT source, ', ')           AS sources,
  CURRENT_TIMESTAMP()                         AS refreshed_at
FROM {D}.auctions_clean
WHERE postcode_sector IS NOT NULL
  AND auction_date IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY 1, 3
''')

run_query('auction_sector_summary', f'''
CREATE OR REPLACE TABLE {D}.auction_sector_summary AS
SELECT
  postcode_sector,
  postcode_district,
  MAX(region)                                 AS region,
  COUNT(*)                                    AS total_lots,
  COUNTIF(is_sold)                            AS total_sold,
  COUNTIF(status = 'UNSOLD')                  AS total_unsold,
  COUNTIF(status = 'WITHDRAWN')               AS total_withdrawn,
  ROUND(SAFE_DIVIDE(COUNTIF(is_sold), COUNT(*)) * 100, 1) AS sell_through_rate_pct,
  CAST(ROUND(AVG(guide_price_mid), 0) AS INT64)  AS avg_guide_price,
  CAST(ROUND(AVG(IF(is_sold, sold_price, NULL)), 0) AS INT64) AS avg_sold_price,
  CAST(APPROX_QUANTILES(IF(is_sold, sold_price, NULL), 100)[OFFSET(50)] AS INT64) AS median_sold_price,
  CAST(MIN(IF(is_sold, sold_price, NULL)) AS INT64) AS min_sold_price,
  CAST(MAX(IF(is_sold, sold_price, NULL)) AS INT64) AS max_sold_price,
  ROUND(AVG(IF(is_sold, premium_over_guide_pct, NULL)), 1) AS avg_premium_over_guide,
  MIN(auction_date)                           AS earliest_auction,
  MAX(auction_date)                           AS latest_auction,
  STRING_AGG(DISTINCT source, ', ')           AS sources,
  CURRENT_TIMESTAMP()                         AS refreshed_at
FROM {D}.auctions_clean
WHERE postcode_sector IS NOT NULL
GROUP BY 1, 2
ORDER BY 1
''')

  Refreshing auction_sector_monthly... done
  Refreshing auction_sector_summary... done


## 8. Pipeline summary

In [11]:
tables = [
    ('ingested_files',        'ingested_at'),
    ('auctions_raw',          'ingested_at'),
    ('auctions_status_log',   'changed_at'),
    ('auctions_clean',        'refreshed_at'),
    ('auction_sector_monthly','refreshed_at'),
    ('auction_sector_summary','refreshed_at'),
]
parts = ' UNION ALL '.join([
    f"SELECT '{t}' AS table_name, COUNT(*) AS row_count, MAX(CAST({ts} AS STRING)) AS last_updated FROM {D}.{t}"
    for t, ts in tables
])
summary = bq.query(parts + ' ORDER BY table_name').to_dataframe()
print(f'=== Auction Pipeline [{ENV.upper()}] ===')
print(f'Dataset : {PROJECT}.{DATASET}\n')
print(summary.to_string(index=False))

=== Auction Pipeline [DEV] ===
Dataset : signalsbi-new.auctions_dev

            table_name  row_count                  last_updated
auction_sector_monthly       6349 2026-06-05 10:00:33.702906+00
auction_sector_summary       3860 2026-06-05 10:00:36.298578+00
        auctions_clean      21312 2026-06-05 09:58:59.475526+00
          auctions_raw      48980 2026-06-05 09:56:08.930974+00
   auctions_status_log        634 2026-06-05 09:56:08.930974+00
        ingested_files          6 2026-06-05 09:56:08.930974+00


## 9. Promote dev → prod
Run manually after validating dev. Set `PROMOTE = True` to execute.

In [ ]:
PROMOTE = False  # <-- set True to copy dev -> prod

if not PROMOTE:
    print('Promotion skipped. Set PROMOTE = True to copy dev -> prod.')
else:
    if ENV != 'dev':
        raise RuntimeError("Set ENV = 'dev' before promoting.")
    prod_dataset = DATASETS['prod']
    to_promote = [t for t, _ in tables]
    for table in to_promote:
        src = f'{PROJECT}.{DATASETS["dev"]}.{table}'
        dst = f'{PROJECT}.{prod_dataset}.{table}'
        print(f'  Copying {table}...', end=' ', flush=True)
        try:
            bq.copy_table(src, dst, job_config=bigquery.CopyJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)).result()
            print('done')
        except Exception as e:
            print(f'SKIPPED ({e})')
    print(f'Promotion complete -> {PROJECT}.{prod_dataset}')